# Congress House Bills Web Scraper

## Description

A web scraper for Congress House bills (HBNXXXXX)

## Imports and Dependencies

### Install Dependencies

In [1]:
!uv pip install -r requirements.txt

Checked 75 packages in 29ms


### Library Imports

In [12]:
from patchright.async_api import async_playwright, Page, Browser
from camoufox.async_api import AsyncCamoufox
import os
import json
import requests
from dotenv import load_dotenv
from datetime import date

### Environment Variables

In [13]:
AWS_BUCKET_DATA_LOCATION = os.getenv("AWS_BUCKET_DATA_LOCATION")
AWS_BUCKET_METADATA_LOCATION = os.getenv("AWS_BUCKET_METADATA_LOCATION")


## Helper and Functions and Class Definitions

### File Class Object Definition

In [14]:
class File:
    def __init__(
            self,
            order_no : str,
            title : str,
            date : str,
            file_link : str,
            date_scraped : str
            ):
        self.order_no = order_no
        self.title = title
        self.date = date
        self.file_link = file_link
        self.date_scraped = date_scraped
files : set[File] = set()

### Global Variables

In [15]:
files : set[File] = set()
changes_counter = 0

### JSON Encoder and Progress Loading for File Object

In [ ]:
def json_encoder(obj: File):
    """
    Encodes FIle class instance to JSON instance 

    Args:
        obj (File): File object

    Raises:
        TypeError: Occurs when object passed is not an instance of the File class

    Returns:
        dict[str,str]: dictionary for JSON parsing
    """
    if isinstance(obj, File):
        return {
            'Order Number' : obj.order_no,
            'Title' : obj.title,
            'Date Filed' : obj.date,
            'File Link' : obj.file_link,
            'Date Scraped' : obj.date_scraped
        }
    raise TypeError("Object is not JSON parsable.")

def load_files_from_json(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            for item in data:
                # Reconstruct the File object using the JSON keys
                # We use .get() to avoid errors if a key is missing
                new_file = File(
                    order_no=item.get('Order Number'),
                    title=item.get('Title'),
                    date=item.get('Date Filed'),
                    file_link=item.get('File Link'),
                    date_scraped=item.get('Date Scraped')
                )
                files.add(new_file)
        print(f"Successfully loaded {len(files)} unique bills.")
    except FileNotFoundError:
        print("No existing JSON found. Starting with an empty set.")

### Download File From URL Function

In [17]:
import os
import re

def to_snake_case(name: str) -> str:
    name = name.lower()
    name = re.sub(r'[^a-z0-9]+', '_', name)
    name = name.strip('_')
    return name

async def download_with_browser(page, url: str, dest_folder: str, return_url: str):
    if not os.path.exists(dest_folder):
        os.makedirs(dest_folder)
    
    try:
        filename = url.split("/")[-1].replace(" ", "_")
        file_path = os.path.join(dest_folder, filename)
        
        response_data = {}
        
        async def handle_route(route):
            response = await route.fetch()
            body = await response.body()
            response_data['body'] = body
            await route.fulfill(response=response, body=body)
        
        await page.route(url, handle_route)
        await page.goto(url)
        await page.wait_for_timeout(2000)
        await page.unroute(url)
        
        if 'body' in response_data:
            with open(file_path, "wb") as f:
                f.write(response_data['body'])
            print(f"Downloaded: {filename}")
        else:
            print(f"Download failed: no body captured")
        
        # Go back to the referring page
        await page.goto(return_url)
        await page.wait_for_selector('tbody > tr', timeout=10000)
        return True
            
    except Exception as e:
        print(f"Download failed: {e}")
        return False


def download(url: str, dest_folder: str):
    return False


## Scraper Stuff

### Actual Scraper

#### File Metadata Reset

In [18]:
# Get updated metadata file from AWS bucket
print(AWS_BUCKET_METADATA_LOCATION)
!rm -rf outputs/*
# !aws s3 cp {AWS_BUCKET_METADATA_LOCATION}metadata.json outputs  

# Populate files array for cross-checking
load_files_from_json('outputs/metadata.json')
changes_counter = 0


None
No existing JSON found. Starting with an empty set.


#### Scraper Code

In [19]:

# List of URLs to scrape
SCRAPE_URLS = [
    ("Administrative Order", "https://ncip.gov.ph/administrative-order/", "tbody > tr"),
    ("Joint Administrative Order", "https://ncip.gov.ph/joint-administrative-order/", "tbody > tr"),
    ("Joint Memo Circular", "https://ncip.gov.ph/joint-memo-circulars/", "tbody > tr"),
    ("Related Issuances", "https://ncip.gov.ph/related-issuances/", "tbody > tr"),
    ("Commission En Banc Resolution", "https://ncip.gov.ph/commission-en-banc-resolution/", "table tbody > tr"),
]
today = date.today().strftime('%Y-%m-%d')

async def scrape_page(page, page_name: str, url: str, row_selector: str):
    """Scrape a single page and download files"""
    print(f"\n=== Scraping: {page_name} ===")
    await page.goto(url)
    await page.wait_for_selector(row_selector, state='visible', timeout=90000)
    
    order_list = page.locator(row_selector)
    order_count = await order_list.count()
    print(f"Found {order_count} entries")
    
    for i in range(order_count):
        order_item = order_list.nth(i)
        order_item_fields = order_item.locator('td')
        field_count = await order_item_fields.count()
        
        if field_count == 3:
            # Standard table with 3 columns (Order No, Title/Link, Date)
            order_no = await order_item_fields.nth(0).inner_text()
            title = ""
            file_link = "None"
            try: 
                title = await order_item_fields.nth(1).locator('a').inner_text()
                file_link = await order_item_fields.nth(1).locator('a').get_attribute('href')
            except:
                try:
                    title = await order_item_fields.nth(1).inner_text()
                except:
                    pass
            if title == "":
                continue
            date = await order_item_fields.nth(2).inner_text()
        elif field_count == 2:
            # Table with 2 columns (Link with title, Description)
            order_no = ""
            try:
                title = await order_item_fields.nth(0).locator('a').inner_text()
                file_link = await order_item_fields.nth(0).locator('a').get_attribute('href')
            except:
                try:
                    title = await order_item_fields.nth(0).inner_text()
                    file_link = "None"
                except:
                    title = ""
                    file_link = "None"
            try:
                date = await order_item_fields.nth(1).inner_text()
            except:
                date = ""
        else:
            continue
        
        if file_link and file_link != "None":
            print(f"  - {file_link}")
            folder_name = to_snake_case(page_name)
            await download_with_browser(page, file_link, f'outputs/{folder_name}/', url)
        
        new_file = File(
            order_no=order_no.strip() if order_no else "",
            title=title.strip() if title else "",
            date=date.strip() if date else "",
            file_link=file_link.strip() if file_link else "",
            date_scraped = today
        )
        files.add(new_file)


try:
    async with AsyncCamoufox(headless=False, geoip=True) as browser:
        context = await browser.new_context(
            viewport={"width":1000, "height":500},
        )
        page = await context.new_page()
        
        # Scrape each URL
        for page_name, url, row_selector in SCRAPE_URLS:
            await scrape_page(page, page_name, url, row_selector)
        
except Exception as e:
    print("Error occurred. Saving progress...")
    print(f"{e}")

# Processing logic (e.g., saving to JSON)
with open('outputs/metadata.json', mode='w', encoding='utf-8') as f:
    json.dump(
        obj=list(files),
        fp=f,
        default=json_encoder,
        indent=4
    )


=== Scraping: Administrative Order ===
Found 21 entries
  - https://ncip.gov.ph/wp-content/uploads/2025/01/AO-01-2024-NCIP-Rules-of-Procedure.pdf
Downloaded: AO-01-2024-NCIP-Rules-of-Procedure.pdf
  - https://ncip.gov.ph/wp-content/uploads/2020/09/ncip_admin_order_01_2020_rules_on_delineation_titling.pdf
Downloaded: ncip_admin_order_01_2020_rules_on_delineation_titling.pdf
  - https://ncip.gov.ph/wp-content/uploads/2025/02/NCIP-AO-1-S.-2021-as-amended-by-CEB-Reso-No.-2023-09-07-063.pdf
Downloaded: NCIP-AO-1-S.-2021-as-amended-by-CEB-Reso-No.-2023-09-07-063.pdf
  - https://ncip.gov.ph/wp-content/uploads/2025/02/NCIP-Administrative-Order-No.-2-series-of-2019-COC-Guidelines.pdf
Downloaded: NCIP-Administrative-Order-No.-2-series-of-2019-COC-Guidelines.pdf
  - https://ncip.gov.ph/wp-content/uploads/2020/09/ncip-ao-no-2-s-2018-revised-adsdpp-guidelines.pdf
Downloaded: ncip-ao-no-2-s-2018-revised-adsdpp-guidelines.pdf
  - https://ncip.gov.ph/wp-content/uploads/2020/09/ncip-ao-no-1-s-2018-rul

### Upload to AWS Bucket

In [21]:
# Data Sync to S3 Bucket
!aws s3 sync outputs/ {AWS_BUCKET_DATA_LOCATION} --exclude "*" --include "*.pdf"

# Metadata upload to S3
!aws s3 cp outputs/metadata.json {AWS_BUCKET_DATA_LOCATION}  



usage: aws s3 sync <LocalPath> <S3Uri> or <S3Uri> <LocalPath> or <S3Uri> <S3Uri>
Error: Invalid argument type

usage: aws s3 cp <LocalPath> <S3Uri> or <S3Uri> <LocalPath> or <S3Uri> <S3Uri>
Error: Invalid argument type
